# RAG end to end: query rewriting

This notebook builds on `Broken-RAG.ipynb`. The only new RAG step is query rewriting: the user's conversational request is turned into a more general information-retrieval query before searching Doug's blog.

In [5]:
from copy import deepcopy
import numpy as np
from IPython.display import HTML, Markdown, display
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from cheat_at_search.data_dir import key_for_provider
from cheat_at_search.doug_blog_data import corpus

openai = OpenAI(api_key=key_for_provider('openai'))

## Search index

In [6]:
def no_chunking(doc):
    yield doc

def chunk_by_paragraph(doc):
    yield from doc.split('\n\n')

def top_n(vectors, query_vector, k=3):
    similarities = np.dot(vectors, query_vector)
    return np.argsort(similarities)[-k:][::-1]

class SearchIndex:
    def __init__(self, corpus, chunk_fn=no_chunking):
        self.corpus = corpus
        self.chunk_fn = chunk_fn
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.all_chunks = []
        self.index = None

    def chunks(self):
        for _, row in self.corpus.iterrows():
            for chunk in self.chunk_fn(row['description']):
                yield row['title'], chunk

    def build_index(self):
        self.all_chunks = list(self.chunks())
        self.index = self.model.encode(
            [chunk for _, chunk in self.all_chunks],
            convert_to_numpy=True,
            show_progress_bar=True,
        )

    def search(self, query, k=3):
        query_vector = self.model.encode([query], convert_to_numpy=True)[0]
        return [self.all_chunks[i] for i in top_n(self.index, query_vector, k)]

## Rewrite conversational requests

The rewrite is intentionally narrow. It removes the request to speak as Doug and produces a general information-retrieval question that should be easier to match against blog content.

In [7]:
answer_system_prompt = '''
You pretend to be search engine expert Doug Turnbull. Answer questions
in his voice using the supplied snippets from Doug's blog. Give Doug's
specific perspective, not a generic answer. If the snippets do not
support a claim, say that the blog context does not establish it.
'''

query_system_prompt = '''
The user is asking Doug Turnbull a question. Reformulate the user's
request as a general information-retrieval question to search Doug's blog.
Remove Doug's name and any request to answer in Doug's voice. Return only
the rewritten search query.
'''

class RAG:
    def __init__(self, corpus, client):
        self.client = client
        self.search_index = SearchIndex(corpus, chunk_fn=no_chunking)
        self.search_index.build_index()
        self.reset()

    def reset(self):
        self.context = [{'role': 'system', 'content': answer_system_prompt}]

    def query_rewrite(self, message):
        response = self.client.responses.create(
            model='gpt-3.5-turbo',
            input=[
                {'role': 'system', 'content': query_system_prompt},
                {'role': 'user', 'content': message},
            ],
        )
        return response.output_text.strip()

    def chat(self, message):
        query = self.query_rewrite(message)
        results = self.search_index.search(query)
        snippets = '\n\n---\n\n'.join(snippet for _, snippet in results)
        context = deepcopy(self.context)
        context.append({'role': 'user', 'content': message})
        context.append({
            'role': 'user',
            'content': "Use these retrieved snippets from Doug's blog:\n\n" + snippets,
        })
        response = self.client.responses.create(
            model='gpt-3.5-turbo',
            input=context,
        )
        answer = response.output_text
        self.context.append({'role': 'user', 'content': message})
        self.context.append({'role': 'assistant', 'content': answer})
        return answer, query, results

In [8]:
prebaked_messages = [
    'What does Doug say is the best way to use BM25?',
    "Explain vector search in Doug's voice.",
]

rag = RAG(corpus, openai)
for question in prebaked_messages:
    answer, query, results = rag.chat(question)
    display(Markdown(
        f'**Question:** {question}\n\n**Rewritten query:** {query}\n\n**Answer:** {answer}'
    ))
    display(Markdown('### Evidence supplied to the model'))
    for title, snippet in results:
        display(Markdown(f'**{title}**\n\n{snippet}'))
    display(HTML("</hr>"))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/30 [00:00<?, ?it/s]

**Question:** What does Doug say is the best way to use BM25?

**Rewritten query:** Best practices for utilizing BM25 algorithm.

**Answer:** The best way to use BM25, particularly in a multi-field scenario, involves understanding the building blocks of BM25 and how to extend it to multiple fields. By utilizing BM25F, you can combine the benefits of BM25 across different fields in a more efficient manner. To effectively implement BM25F, you need to consider factors such as per-field BM25Similarity configuration and the use of BlendedTermQuery to handle document frequencies across multiple fields. Additionally, normalizing the lexical score from BM25 to a range of 0-1 can offer benefits, even though it involves some trade-offs. Understanding the intricacies of BM25 and its extension to multiple fields can greatly enhance search relevance and performance.

### Evidence supplied to the model

**BM25F in Lucene with BlendedTermQuery**

 As part of the London hack days Diego Ceccarelli started a BM25F implementation. I began to continue it at Lucene Revolution’s Lucene hackathon. I realized though that when you break down the problem, BM25F can be implemented using existing Lucene bits, including the existing BM25Similarity and the BlendedTermQuery. 

 BM25 and BM25F 

 I’ve written about BM25 before. BM25 is an iteration on the classic TF*IDF ranking formula, but with several innovations: 

 
   Term Frequency Saturation: We know more times a search term occurs in a document, the more we should consider this document relevant for that search term. But with BM25, we realize at some point you reach diminishing returns. BM25 reaches this point relatively quickly, compared to classic TF*IDF. 
   Smarter document length weighting: A search term occurring once in a short doc is more relevant than a single term occurring in a longer doc (a book). BM25 penalizes/rewards document length relative to a document’s average document length, as opposed to just having a constant multiple based on document length 
   Basically the same IDF calculation: Rare search terms (low doc freq; high IDF) get weighted more heavily than common search terms. The BM25 calculation isn’t that much different than TF*IDF’s calculation 
 

 BM25F performs per-field BM25 calculation, but uses the shared document frequency across multiple fields. This document frequency blending is important. For example, sometimes you have scenarios where a common term, say “cat” is actually rare in one particular field (say the “title” field). But when you broaden out to other fields, you then realize that cat is particularly common, and shouldn’t be scored so highly. 

 In other words, BM25F basically does 

 CombinedIDF * ( BM25_title + BM25_description + …)
 

 (note there are all kinds of subtle variants) 

 Can we implement that using existing Lucene bits? I think so… with a few caveats that Diego has pointed out. You can follow along by looking at this github repo with the sample code from this post. 

 Per-field BM25Similarity 

 BM25F lets us configure BM25 parameters per field. Luckily, per-field similarity is pretty easy to configure in Lucene using a PerFieldSimilarityWrapper. We simply need to setup our index accordingly. Notice in the similarity below, k1 and b differ for title and description: 

 java
static Similarity perFieldSimilarities =  new PerFieldSimilarityWrapper() {
      @Override
      public Similarity get(String name) {
          if (name.equals("title")) {
              return new BM25FSimilarity(/*k1*/1.2f, /*b*/0.8f);
          } else if (name.equals("description")) {
              return new BM25FSimilarity(/*k1*/1.4f, /*b*/0.9f);
          }
          return new BM25FSimilarity();
      }
};
 

 Then when we setup the IndexReader 

 ```java
IndexWriterConfig config = new IndexWriterConfig(analyzer);
config.setSimilarity(perFieldSimilarities); 

 IndexWriter w = new IndexWriter(index, config);
```` 

 BlendedTermQuery to implement BM25F 

 Lucene’s BlendedTermQuery forms the guts behind Elasticsearch’s cross_field search. What BlendedTermQuery does is blend global term stats as best as possible across different fields. If the document frequency for cat in title is 3, but the document frequency for cat in the description field is 50, it takes the maximum, 50. This approximates the actual document frequency across both fields (we can’t figure out overlaps very efficiently at query time). 

 This value is then used for searching both title and description, accounting for the true rareness of the term. To do this, it builds a set of Lucene TermQueries and tells the TermQuery for each field to use the blended document frequency, instead of what’s in the index for that field. So you can imagine that now you have two queries description:cat (\*with doc freq 50) and title:cat (\*with doc freq 50) 

 BlendedTermQuery then gives you two options for combining these queries. You can either as a dismax query (take the max of the underlying field scores) or a boolean query (sum the underlying field scores). In other words, you can pick between one of two ranking functions: 

 
   Dismax: The best scoring field wins, with an optional tie-breaker parameter to incorporate other field scores. In other words: max( description:cat (with doc freq 50), title:cat (with doc freq 50), …) 
   Boolean: A summation of the field scores, in other words description:cat (with doc freq 50) + tilte:cat (with doc freq 50) + … 
 

 Examining the math above, we really care about the latter form for BM25F as it involves a summation. To implement that, we simply use the following code. 

 ```java
BlendedTermQuery query = new BlendedTermQuery.Builder()
				 .add(new Term(“title”, “cat”), /boost/1.0f)
				 .add(new Term(“description”, “cat”), /boost/1.0f)
				 .setRewriteMethod(BlendedTermQuery.BOOLEAN_REWRITE)
				 .build(); 

 ``` 

 Checking the ranking function 

 So with the BlendedTermQuery above, what we have now is something like the following ranking function 

 description:cat (with BM25, docfreq=50) + title:cat (with BM25, docfreq=50)
 

 Which is the same as 

 IDF( docfreq=50) * (description:cat with BM25) +  IDF(docfreq=50) * (title:cat with BM25) + 
 

 Here description:cat with BM25 means the non-IDF portion of the BM25 calculation (the term frequency and length part, specific to this field, boosts for this field, etc). The IDF(docfreq=X) is the BM25 IDF formula for a term with a given document frequency. 

 Now it turns out, this is (pretty much) BM25F! All we need to do to make this BM25F is to factor the IDF (docfreq=50) out to see the formula we have above 

 IDF( docfreq=50) * ( (description:cat with BM25) +  (title:cat with BM25) )
 

 A few subtle differences 

 There’s a few reasons this isn’t quite BM25F. First of all Lucene’s boolean query uses a coordinating factor (coord) to reward/punish documents that match all the clauses. So: 

 IDF( docfreq=50) * ( (description:cat with BM25) +  (title:cat with BM25) )
 

 Is actually 

 coord * IDF( docfreq=50) * ( (description:cat with BM25) +  (title:cat with BM25) )
 

 If you recall coord is the number of matched clauses / number of total clauses. So if only 1 out of 2 clauses match, coord is ½. This further rewards documents that match more than one clause. Latest versions of Lucene have removed coord 

 Another thing to point out is that it’s difficult to piece together the true cross-field document frequency at query time. So the maximum is taken as an approximation. The actual blended document frequency isn’t entirely accurate, but we know it ranges somewhere between the max of the two field doc frequencies (when there’s 100% overlap) and the sum (when the fields mention the term in different documents). 

 And that’s it! Get in touch. 

 That’s it! Get in touch – I’d love to hear feedback if I missed anything. And if you want to pick my brain for free check out the lunch and learns. I’m about to announce some new topics that might interest your team! And as always, don’t hesitate to reach out about our relevancy services – we just got this great testimonial from Careerbuilder about our services: 

 
   OSC’s Solr/Lucene knowledge expanded and greatly improved the abilities of our public job search. They delivered technical excellence at every turn: demonstrating expertise in Lucene internals, relevance models, and data science backed by a solid methodology for improving search relevance. On a deliverables front: they learned our legacy search stack quickly and made high-quality code contributions. OSC marries technical excellence with strategic insight: we highly recommend their experts to any search team. 
 


...

**Question:** Explain vector search in Doug's voice.

**Rewritten query:** Find information on vector search on the blog.

**Answer:** Vector search, as described by Doug in his blog, involves working with arrays of numerical values representing embeddings. These embeddings can be from a wide array of sources like words, sentences, images, etc., and the goal is to find similar vectors efficiently in a vast dataset. Doug explains that vector search relies on advanced data structures like Hierarchical Navigable Small Worlds (HNSW) to enable the retrieval of the nearest neighbors to a given query vector efficiently.

Doug goes on to explore various concepts related to optimizing vector search, such as quantization to adjust for data distribution sensitivity, clustering techniques for grouping data points, and the use of graphs in algorithms like HNSW. By explaining the process of constructing a graph for vector search and providing code snippets illustrating the implementation in Python, Doug offers insights into the practical application of vector search in a search system.

Furthermore, Doug veers into discussions about the proliferation of vector databases, the evolving landscape of retrieval problems beyond simple vector storage, and the necessity for a holistic approach to building retrieval and ranking systems that address a wide array of user needs. He delves into topics like lexical scoring with BM25, the importance of understanding query processing and intent classification, and the integration of machine learning techniques like Learning to Rank for optimizing search results.

Overall, Doug's comprehensive exploration of vector search encompasses not only the technical aspects of working with vectors and nearest neighbor retrieval but also delves into the broader context of the evolving search ecosystem and the challenges and opportunities in modern retrieval systems.

### Evidence supplied to the model

**On the road to your own vector db - some basics**

On Wednesday May 7th, I'll stream myself [live-coding a vector database](https://maven.com/p/866f13/doug-live-codes-a-vector-database) with [John Berryman](https://arcturus-labs.com/). As part of that, I want to establish some baseline ideas / concepts before digging into the most popular vector search data structure - [Hierarchical Navigable Small Worlds](https://t.co/C4HkmUZeMA) (HNSW). HNSW sits at the core of systems like Weaviate, Elasticsearch, Vespa, OpenSearch, PGVector, and many others.

##  What are vectors / vector search?

Vectors are just arrays of length N, where N might be 768, 1024, 1536, etc. We usually say "N dimensions". We want to take a query vector (ie `[1,2,3,…]`) and find the K closest vectors in our system (ie maybe `[1.1, 1.9, 2.9,…]`). 

These vectors correspond to [vector embeddings](https://cohere.com/blog/embeddings), a representation of a word, sentence, image, or, really anything. Embeddings come out of models that move similar items closer. Our model might know that "Mary had a little lamb" is very similar to "Little bo peep had a sheep" - yielding nearly identical embeddings - despite sharing no important words.

At scale, we need a way to enter an embedding for "Mary had a little lamb" `[1,2,3,...]` and pull back "Little bo peep had a sheep" `[1.1, 1.9, 2.9]` amongst the millions/billions of other vectors. That's what vector search does.

We use a lot of two-dimensional analogies for high-dimensions (which [can be imperfect](https://softwaredoug.com/blog/2022/12/26/surpries-at-hi-dimensions-orthoginality)). But here, I think they're apt. I'll abuse Latitude / longitude on a map as one example of a 2D "vector".

## Vector search that doesn't work

Imagine if Lat / Long corresponds to property. Maybe you want to find the closest parcels a given location. Well that's just 2D vector search.

Your first idea might be to chop up the space - such as on a grid. We find some “region” where `[1,2,3,..]` might exist. Then return all the parcels in that region. For our map analogy, this might be a grid like the US National Grid below (screenshot from [arggis](https://www.arcgis.com/home/item.html?id=d96095fb637846889fb0e46ce69e3967)). If you wanted to find nearest neighbors to lat / long 40.7128° N, 74.0060° W (New York City) you’d zoom in to grids 18T and then subgrid 18TWL like below:

![image.png](/assets/media/2025/hnsw-grid1.png)

![image.png](/assets/media/2025/hnsw-grid2.png)

Unfortunately, we know zooming into 18TWL will give us millions of parcels. Many of which are ***very similar*** to 40.7128° N, 74.0060° W but ***may not be nearest neighbors***.

That’s the first lesson of nearest neighbor search **nearest neighbors is NOT about finding ‘all the candidates within some nearby distance’** - its instead about ***nearest neighbors***. Depending where you live, your neighbor might be feet or miles away.

We all know New York is incredibly dense, finding nearest neighbors requires careful precision. Given that, could we just make a bigger grid? Well a super deep grid down to won't serve a place like Wyoming - you'd have to gather many many granular squares until you got top K closest parcels. When you get away from 2D analogies, and have N dimensions the problem becomes tougher, creating more degrees of freedom where something can be near or far.

For any vector space - **accurate search requires sensitivity to the distribution of data points**. You have to understand the actual layout of people/property on your map.

## Clusters, graphs, quantization

**Quantization -** Some vector search systems _quantize vectors_ to add sensitivity to the data distribution. Quantize usually means taking a very precise number like 40.7128 and shrinking to 40.7 to save space/time but sacrificing accuracy. But to be sensitive to the data, quantization, also means squishing or stretching lat/long to a different range to accomodate the actual population density, like the cartogram below (taken from [NHGIS](https://commons.wikimedia.org/wiki/File:CartogramPresidentialCounty1964Colorbrewer.gif)).

![image.png](/assets/media/2025/hnsw-cartogram.png)

Imagine we squished lat/long to an 8 bit integer. Some mathematical wizardry has helped put New York’s spaces into values 0-32 of each dimension. Los Angeles, gets 33-48 of that dimension. Wyoming just gets a few measly bits. You can dig into [product quantization](https://www.pinecone.io/learn/series/faiss/product-quantization/) and [better binary quantization (BBQ)](https://www.elastic.co/search-labs/blog/better-binary-quantization-lucene-elasticsearch) for how this works.

**Clusters -** Another - very related - idea is to cluster data points with combinations. Such as the image below showing k-means population clustering (image below by [Michael Gastner](https://www.researchgate.net/figure/Facility-locations-determined-by-simulated-annealing-and-the-corresponding-Voronoi_fig1_6880647))

![image.png](/assets/media/2025/hnsw-voronoi.png)

K-means starts with some K points (like 500? 1000?) and nudges around them around the map until each minimizes its total distance to the actual data points (ie parcels of land).

Once trained, you can imagine having a list of centroids to lookup with a lat long - that points to the rough “neighborhood”. Then at least instead of a constant-zide grid, you’re working with a tiny zone that might be a slice of New York... or the entire state of Wyoming.

Clustering has long been part of vector search hacks [from the early days](https://www.youtube.com/watch?v=bWc4DOM-CHk), as they fit somewhat cleanly into traditional search indices inverted index data structure. Each cluster can point to a list of (vectors, people, parcels, etc) that live there. File formats like [IVF also map centroids to clusters](https://docs.oracle.com/en/database/oracle/oracle-database/23/vecse/understand-inverted-file-flat-vector-indexes.html) in an inverted index style format. Clusters drive Billion-scale vector indices like [SCANN/SPFresh](https://arxiv.org/pdf/2410.14452) - used by systems like [turbopuffer](http://turbopuffer.com). 

**Graphs -** But we’re here to talk about graphs. That's what HNSW uses. 

If we connect all the neighbors together, and walk the graph until we get to the lat/long gets closest to our NYC coordinates, we'll eventually gather that coordinates neighbors. Perhaps like the **roadmap:** below

![image.png](/assets/media/2025/hnsw-roadmap.png)

In this map we start somewhere, and head in the direction of our destination. If we encounter a fork in the road, we see where each fork heads, and choose the path that gets us closer to our destination. We track where we’ve visited, as to avoid loops.

This, indeed works!

## Let's see it in code

Map based analogies only get us so far!

To fulfill the promise of building HNSW we need a graph. And every graph has a node:

```python
class Node:
    def __init__(self, vector, id):
        self.vector = vector
        self.id = id
        self.neighbors = []
```

And a graph holding some nodes:

```python
class VectorGraph:
    """Fully connected vector graph."""
    def __init__(self):
        self.nodes = []
```

We'll write search first:

```python
  def _search_graph(self, vector, k):
      visited = set()       # IDs of nodes visited thusfar
      last_num_visited = 0
      nearest_neighbors = []    # Nearest neighbors thusfar
      entry_point = self.nodes[0] if self.nodes else None     # Entry point
```

A lot of this is standard “walking a graph” stuff a Computer Science undergrad can do. We track where we've been to avoid double work. We always start with a consistent entry point (otherwise, when we use this search to build the graph, some roads won't connect to each other!).

Any time we encounter a new Node in the graph we attempt to save it:

```python
  def _save_neighbor(node):
      if node.id in visited:
          return
      distance = euclidean_distance(vector, node.vector)
      heapq.heappush(nearest_neighbors, (-distance, node))
      if len(nearest_neighbors) > k:
          heapq.heappop(nearest_neighbors)
      visited.add(node.id)
```

Using Python’s heapq library, we maintain a heap of the closest items, popping off anything past the top k. We also track the ids visited to not revisit anything.

Now we visit / add our current node’s neighbors, and move to the closest neighbor node we know about so far. Finally we stop our loop.

```python
  while True:
      # Save all the current search nodes neighbors if close enough...
      for neighbor in curr.neighbors:
          _save_neighbor(neighbor)
      # Stop if nothing changed...
      if len(visited) == last_num_visited:
          break
      # Get the closest neighbor...
      curr = heapq.nlargest(1, nearest_neighbors)[0][1]
			last_num_visited = len(visited)
 return heapq.nlargest(k, nearest_neighbors)
```

Finally to add a node, we just search for the best insert location, and insert accordingly

```python
  def add(self, vector, id):
      node = Node(vector, id)
      if not self.nodes:
          self.nodes.append(node)
          return
      nearest_node = self._search_graph(vector, 1)
      self.nodes.append(node)
      if nearest_node:
            dist, neighbor_node = nearest_neighbors[0]
            # Connect to the nearest neighbor
            neighbor_node.accept_neighbor(node)
            node.accept_neighbor(neighbor_node)
```

(All this obviously non-production ready code can be [found here](https://github.com/softwaredoug/hnsw/tree/main/hnsw))

### Real world considerations

This is a fairly straight-forward example of walking a graph to get closer to a destination. But we might consider a few things:

1. Limiting the number of neighbors any node can have. Right now that’s unbounded
2. Connecting our graph to more than just one node. Right now we connect to exactly 1 neighbor. This will make the search potentially less accurate
3. We only search exactly one neighbor node, what if the best route is some side route from the second closest neighbor from this node?

These relate to two [hyperparameters we’re not paying attention to yet](https://github.com/nmslib/hnswlib/blob/master/ALGO_PARAMS.md#hnsw-algorithm-parameters) - ef and M:

* ef - controls how many paths we go down our roadmap to find possible neighbors
* M - how connected our graph is

### And the big one - hierarchy

Even with a well-connected graph, we have a problem - very little hierarchy. If every road is a surface road in the US, finding directions can be a nightmare. If you're driving, you hope to use the “higher level” constructs (boulevards / highways / interstates / etc) to efficiently navigate between two major cities. Otherwise navigating between LA to New York would involve 1000s of tiny turns, making the road-graph nearly useless. Highways let you quickly jump between regions, avoiding thousands of twists and turns. That's where the _hierarchical_ of HNSW comes from.

### See you Wednesday

But these finer points, and more (like adding hierarchy!), will need to wait till my embarrasing attempt to [live-code all of this on Wednesday](https://maven.com/p/866f13/doug-live-codes-a-vector-database). Hope to see you there!

**Are we at peak vector database?**

As both a search person and something of a veteran of the NoSQL days, I wonder to myself, often, “how can so many vector databases need to exist?”.

Even in our current AI age, where everyone and their mom is trying to build the next chat / AI powered whatever. Even today when it seems everyone is putting vectors somewhere to retrieve them…

I have to ask the tough question - when have we reached peak vector DB? Do we have enough choices for the specific task of storing and retrieving vectors?

Just on cursory listing, I can think of the following vector databases, libraries, whatever:

**Pure Vector DBs**
* Pinecone
* QDrant
* Milvus / Zilliz
* Weaviate
* Turbopuffer
* MyScale

**Open Source search engines**

* Solr
* Elasticsearch
* OpenSearch
* Vespa

**Libraries**

* Annoy,
* FAISS,
* NMSLib
* HNSWLib
* Lucene
* Chroma
* (a million others)

**Open Source DBs**

* Redis
* PGVector
* Cassandra, Mongo, etc etc (every DB seems to be getting its vector index :wink: )

**Cloud solutions**

* Google Vertex
* Azure AI Search

(more at [awesome vector search](https://github.com/currentslab/awesome-vector-search))

If we take vector search as one type of data store in the NoSQL paradigm, we might put it in its own category. We would say Mongo is a document database alongside CouchDB and pals. We would say Cassandra is a columnar data store, alongside the Scylla or HBase. 

So in each category, we have a handful (2-3). Yet in vector search, we have dozens upon dozens of options. As a “customer” of such options, the field becomes overwhelming.

And vector retrieval increasingly isn’t the problem. The hard problems of solving real-world retrieval are not related to just getting vectors, it’s everything around it

Intent classification - given a “query” how do I know whether I can solve the problem or not (RAG) or how to route the query to the correct place
Inference and reranking - given a “query”, and some candidate retrieved vectors / items, how do I perform inference on say, an arbitrary tensorflow model, to give the most relevant items?
Diversity - given a “query” how do I broaden the candidate pool to more than just “similar to vectors” - to get at not just one intent, but all possible intents
Lexical retrieval / hybrid search - given natural language, how do I use direct lexical signals (boring old BM25, just filtering, whatever) to give relevant results

OK and that’s just the lexical side. We’re inventing new ways of interacting with data. Nobody I talk to has really created a robust way to evaluate quality of context for RAG. There’s new UX paradigms out there - chat and chat-adjacent - that we’re experimenting with.

The challenge being, the world is wide open for experimentation, yet on first blush, all the money is being concentrated in one part of the stack. We’re not looking at the problem holistically.

## Why we’re not at peak vector DB

OK, that’s one argument, sure. 

Here’s the other point of view.

There needs to be a place to focus on, and rethink, retrieval problems. In the same way NoSQL forced us to rethink databases. Capital and brainpower need a place to zero in on how to solve this next generation of retrieval + relevance problems.

So the old, curmudgeonly search person in me would say “well whatever, people realize they need search engines and use Solr / Elasticsearch anyway”. 

But that’s not good enough. These search tools feel esoteric, in the average “AI Engineers” mind, they will think first of vector retrieval, then stumble into all the problems I list above. They’ll learn they need to care about all the things beyond ANN, but only after their app is stood up. In the same way the search engineers of yore backed into all kinds of problems only after comitting to a big Solr or Elasticsearch installation.

Additionally, I suspect, more and more surfaces will be driven by some retrieval-ish thing. Search-but-not-search. Real-time recommendations, but driven by vector (and other kinds of) retrieval that looks more like a search engine -  not batch computed, nightly jobs common these days. I wrote about the coming revolution of “Relevance Driven Applications” in 2016, and now, its happening - not with boring old search engines as I once thought - but by reinventing our whole notion of the retrieval layer that drives user experiences.

So, I suspect the smart people at these companies will branch out beyond “making ANN better / more scalable” to building complete retrieval and ranking systems solving a tremendous array of problems. The money and effort will flow to where the customers see the problem.

In the end, like in NoSQL, we may end up with SQL again (but with all the NoSQL innovations). Or, in other words, we may end up with these vendors building a full blown “search engine”. Yet these new search engines will have many more batteries included the AI/Chat/RAG/whatever experiences people increasingly reach for.

**What AI Engineers Should Know about Search**

Things AI Engineers Should Know about Search. At least 50 of them :).

I probably don't need to discuss bi/cross-encoders, etc. A lot of great content is out there on those topics, especially folks like [Pinecone](https://www.pinecone.io/blog/), targeting the AI / LLM / RAG crowd. But maybe you to quickly get some high-level, historical, lexical search context... Well I'm here for ya! You might be new to all this. Hired into an "AI Engineer" role and suddenly needing to index a lot of search content :). Welcome to this brave world, there's a lot you can contribute!

Some things to know:

1.  The fanciest solutions don't matter as much as getting a good evaluation framework setup to evaluate the quality of search results

2.  Your choice in Vector Database probably doesn't matter as much as how you fit all the pieces of a retrieval solution together - lexical, vector, reranking, query understanding, etc. More people seem to be aware of this now, and we've probably passed [peak vector DB](https://softwaredoug.com/blog/2024/01/24/are-we-at-peak-vector-db).

3.  There are many ways to label search results as relevant / not. Whether crowdsourcing human labelers, to processing clickstream data, to having an LLM evaluate search results using a prompt. They all have their pros / cons.

4.  Search labels are often called [judgments](https://softwaredoug.com/blog/2021/02/21/what-is-a-judgment-list) - because historically because a "judge" would give a "grade" (relevant or not) to a search result for a query back in the first Information Retrieval competitions like TREC.

5.  Evaluation methods all have[ different biases](https://jamesrubinstein.medium.com/measuring-search-a-human-approach-acf54e2cf33d). In human evaluators we have issues dealing with non-experts, people getting exhausted, people preferring earlier results as more relevant than later results, etc

6.  In clickstream based judgments, we have other issues! We only get labels for results the search system returns in the top N (stuff that can be clicked on). And we deal with people's lizard brains clicking on irrelevant, spicy pictures.
  
8.  <a href="https://softwaredoug.com/blog/2022/07/16/what-is-presentation-bias-in-search">Presentation bias</a> in search UIs is a big problem. Basically users only click on what they see!

9. Overcoming presentation bias involves an active or reinforcement-learning mindset - where we "exploit" the current knowledge of search relevance, but find ways to "explore" new results. A great resource for overcoming this is <a href="https://livebook.manning.com/book/ai-powered-search/chapter-12/v-20">Chapter 12 of AI Powered Search</a>

10. Click models can turn clickstream data into judgments. There's a great book <a href="https://clickmodels.weebly.com/uploads/5/2/2/5/52257029/mc2015-clickmodels.pdf">Click Models for Web Search</a> available for free that can guide your work understanding user search result clicks and conversions.

11.  A lot of metrics exist for measuring the quality of a query - if query "zoolander" returns some search results, we can reference the judgments, to see whether or not we gave relevant results. Statistics like (n)DCG, ERR, MAP, Precision, Recall, F-Score are well understood in the search industry
    
13. Search comes with a [precision / recall tradeoff](https://link.springer.com/referenceworkentry/10.1007/978-0-387-30164-8_652#:~:text=Precision%20%3D%20Total%20number%20of%20documents,to%20define%20these%20two%20measures.). If you cast a wide net, you'll get more relevant results in the mix, but also likely be showing the users lots of irrelevant ones too!

14.  Search people talk about an "Information Need" some informal specification of the information a user wants. They express that information need with  multiple queries

15.  Information needs can be quite diverse - from looking for an exact item by ID to comparing / contrasting products to performing in-depth research on one topic and gathering notes, they each can require unique treatment in the ranking and UX

16.  How users search is only getting more complicated. Users have very different expectations of question answering systems, social media sites, e-commerce, RAG, etc.

17.  Maybe you've heard of [BM25](https://opensourceconnections.com/blog/2015/10/16/bm25-the-next-generation-of-lucene-relevation/). It is "just" [TF*IDF](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) that's been heavily tuned and studied over the years.

18.  BM25 saturates the term-frequency component of TF*IDF, the intuition being that relevance and term frequency aren't linearly related, but after enough mentions of the term, it doesn't suddenly get "more about" that topic

![BM25 term saturation](/assets/media/2024/bm25_tf.png)

<ol start="17">
<li><p>IDF - or Inverse Document Frequency (1/doc freq) - measures the term's specificity the rarer the term, the more important it is to the user's intent. In a search for [luke] OR [skywalker] - skywalker is rarer and more specific so it scores higher.</p></li>

<li><p>IDF is not actually raw IDF - 1 / docfreq, but each similarity system has its own IDF curve that saturates non linearly</p></li>

</ol>

<img src="/assets/media/2024/idf.png">

<ol start="19">
  
<li><p>BM25 also biases towards shorter fields. Because the probability of you mentioning "Luke Skywalker" in a tweet is much much less than it happening to crop up once in a long textbook.</p></li>

<li><p>BM25 can be <a href="https://stackoverflow.com/questions/38071877/how-to-choose-the-okapi-bm25-parameters-b-and-k1">tuned with parameters k1 and b</a>. The parameter k1 controls saturation of term frequency - such as very fast for short snippets of text, or very slow for longer snippets</p></li>

<li><p>The parameter b changes the influence of field length on the score. Set b very high and above average fields will be punished more (below average punished less).</p></li>

<li><p>BM25 is just one of a bazillion lexical similarity scores. Elasticsearch, for example, has <a href="https://www.elastic.co/guide/en/elasticsearch/reference/current/index-modules-similarity.html">many options</a> - including scripting the similarity!</p></li>

<li><p>BM25F is a variant of BM25 that accounts for matching in different fields (with their own term statistics). Field here meaning different text attribute like title or overview or tags or somesuch.</p></li>

<li><p>In many lexical search engines, you may be surprised when a common term in one field, is actually rare in another, screwing up BM25 calculations and suddenly pushing a seemingly irrelevant document to the top.</p></li>

<li><p>There are tools to compute the "true" specificity of a term beyond direct IDF, such as <a href="https://www.elastic.co/guide/en/elasticsearch/reference/current/query-dsl-multi-match-query.html">blending IDF across fields</a>, or merging all the text into one big field.</p></li>

<li><p>Combining and weighing different fields is important. Fields in search play different roles, and need to be tokenized and scored differently. Like prioritizing a title match over something buried deep in a body</p></li>

<li><p>Search exists in an ecosystem. Some fields may be prone to abuse by users (ie keyword stuffing). Some fields have strong incentives to be reasonably non-spammy (like the title). How you score each can be very dependent on how much we can trust the curation of that content.</p></li>

<li><p>Search exists in an ecosystem - one of the best search hacks is to convince our users to SEO their content for search. They'll tune their content towards our search, rather than us needing to tune their algorithm for their bad content.</p></li>

<li><p>Search exists an an ecosystem - one of the worst search hacks is when our users realize they can SEO their content for our search, and they tune / abuse their content to be spammy and nearly malicious</p></li>

<li><p>Tokenization matters a lot. Not just in lexical search, but embeddings too! Tokenization (n-gram, word-piece, whole tokenization) and related topics like stemming, dealing with punctuation, etc can dramatically change how well search performs</p></li>

<li><p><a href="https://blog.mikemccandless.com/2012/04/lucenes-tokenstreams-are-actually.html">Lexical token streams are really graphs</a> - in any position an equivalent token (synonym, or just a different way of expressing tokenization) is just a branch on this graph. And different branches might have different lengths! (IE USA vs United States of America).</p></li>

</ol>

<img src="/assets/media/2024/token_graph.png">

<ol start="32">

<li><p>It's not just about term matching - phrases matter a lot. Phrases correspond to important entities users often search for, or tags people stick on things, or many other things we expect. This really matters in very technical disciplines like science, engineering, or legal research - where that phrase has some very specific meaning.</p></li>

<li><p>Phrases require encoding term positions - It's not feasible to build a giant lexical term dictionary of every term combination. So a lexical search system has to encode positions of terms in a data structure like a <a href="https://softwaredoug.com/blog/2024/01/21/search-array-phrase-algorithm">roaring bitmap</a></p></li>

<li><p>As phrases need to be tokenized - and might produce equivalent terms at a given position - they are also graphs! If you expect USA to be a synonym of United States of America, that's a graph! A good query parser has to treat a search for USA as actually a search for "USA" OR "United States of America"</p></li>

<li><p>Some term-bigrams statistically go together. Think "Palo Alto" or "Nacho Cheese". These are known as <a href="https://opensourceconnections.com/blog/2019/05/16/unreasonable-effectiveness-of-collocations/">collocations</a> and can be great "poor man's" entities.</p></li>

<li><p>Lexical search isn't just about the BM25, but building phrase search in a way that respects the graph-based nature of the token stream</p></li>

<li><p>Usually search systems are a series of pre-processing (query/intent understanding) and post-processing (reranking / post-filtering) stages. With one or more retrieval engines in between.</p></li>

<li><p><a href="https://queryunderstanding.com/">Query Understanding</a> comes in a lot of flavors. From broad, coarse-grain category classification (this is a search for an image) vs extracting entities from a query (this is a search about "San Francisco") vs mapping a query to a vector for vector retrieval</p></li>

<li><p>Query Relaxation is a technique where you first try with a strict, high precision query. Then if you get no / poor results you reissue the query with a relaxed query. A simple version is switching from an OR query (match any terms) to an AND query (match all terms). But its an active area of research - <a href="https://haystackconf.com/2019/query-relaxation/">here's an interesting talk on the subject.</a></p></li>

<li><p>Related to query relaxation is the idea of <a href="https://livebook.manning.com/book/relevant-search/chapter-7/156">relevance 'tiers'</a>. High precision queries are boosted higher, and the lower precision / higher recall strategies are given less weight. If the high precision query doesn't match, you effectively fall back to the lower quality/broader results.</p></li>

<li><p>You can use <a href="https://www.elastic.co/guide/en/elasticsearch/reference/current/query-filter-context.html">filter queries</a> to express what definitely is NOT relevant so that you can exclude those from the results. This helps eliminate false positives from creeping in top N.</p></li>

<li><p>Applying machine learning to optimize ranking is often called "Learning to Rank". As they involve a list-wise ranking relevant results above irrelevant results, they can be a bit of a different type of machine learning than classification or regression.</p></li>

<li><p>Learning-to-Rank functions have a long history of study. <a href="https://docs.google.com/presentation/u/0/d/1UI4KHWDP0hONl21i1tyvvDyCVrgcooojdxYXvGhlPBc/edit">SVMRank</a> is a simple support vector machine that classifies relevant vs irrelevant pairs using a binary classifier SVM. <a href="https://www.microsoft.com/en-us/research/uploads/prod/2016/02/MSR-TR-2010-82.pdf">LambdaMART</a> attempts to optimize a list-wise relevance metric (like NDCG) using gradient boosting</p></li>

</ol>

<img src="/assets/media/2024/svmrank.png">

(Image from <a href="http://aipoweredsearch.com">AI Powered Search</a>)

<ol start="44">

<li><p>LambdaMART is one of those things like BM25 that's old school, but works, and you probably should learn. Here's a <a href="https://softwaredoug.com/blog/2022/01/17/lambdamart-in-depth">good blog post</a> complete with notebooks from scratch :)</p></li>

<li><p>Something like LambdaMART is great when there are a lot of lexical tabular features. Like statistics you want to use in ranking.</p></li>

<li><p>LambdaMART is great for reranking hundreds or thousands of results relatively inexpensively.</p></li>

<li><p>The relationship between relevance and the features are very non-linear. Some things like recency might matter if searching for news. Other things may matter when searching for canonical information. Even a simple relationship between relevance and title / overview BM25 for movie search below is complicated:</p></li>

</ol>

<img src="/assets/media/2024/non_linear_relevance.png">


(Image from <a href="http://aipoweredsearch.com">AI Powered Search</a>)

<ol start ="48">

<li><p>In Learning to Rank training data, a common hack is to sample other query's positive results as negative examples for this query - similar to <a href="https://www.v7labs.com/blog/contrastive-learning-guide">contrastive learning</a></p></li>

<li><p>It's pretty common to have multiple reranking stages. Starting with dumb rerankers optimized for top 1K up to fancy cross encoders that rerank small amounts of search results.</p></li>

<li><p>Popular search systems (Solr, Elasticsearch, Vespa, OpenSearch) have differing levels of functionality for gathering Learning to Rank features, storing, and performing inference on ranking models</p></li>

<li><p>Good features in Learning to Rank systems are both orthogonal from existing features and add value. Orthogonal implies they measure different things, and probably come from different systems (like BM25 vs embeddings vs ??) almost guaranteeing there's no one size fits all solution to search</p></li>

<li><p><a href="https://x.com/HanchungLee/status/1805325078076473733">Ranking and similarity are quite different things!</a> - Two pieces of text can be similar (measured with an embedding, BM25, or otherwise) but still be spammy, old, or otherwise low value.</p></li>

<li><p>Sometimes you don't need a reranker! A lot of people get by just by optimizing the scoring of the first pass using techniques like genetic algorithms or <a href="https://haystackconf.com/us2022/talk-5/">Bayesian optimization</a></p></li>

<li><p>It's OK to cheat! A common technique is to build a collection, like a <a href="https://livebook.manning.com/book/ai-powered-search/chapter-8/v-20">signals collection</a> that memorizes engaging results for a query. Then when a user searches you can upboost highly engaging content (or downboost stuff that's had its chance).</p></li>

<li><p>Query feedback can be fed into your system! - you can learn a lot about your queries by following the types of content that are interacted with. Such as the category they tend to go with, or the embedding of an image that tends to get clicked on. Learn about <a href="https://en.wikipedia.org/wiki/Relevance_feedback">Relevance Feedback</a> </p> </li>

<li><p>A lot of tools exist to prototype. From <a href="http://github.com/softwaredoug/searcharray">SearchArray</a> to <a href="https://github.com/xhluca/bm25s">BM25S</a> you can kick the tires of different lexical solutions to your problems!</p></li>

<li><p>Read <a href="https://nlp.stanford.edu/IR-book/information-retrieval-book.html">Introduction to Information Retrieval</a> by Manning et. al. It's the search bible and covers the foundational concepts of search and retrieval.</p></li>

<li><p>Be sure to check out the many resources about search and information retrieval, one great list is at <a href="https://github.com/frutik/awesome-search">Awesome Search</a> Github repo</p></li>

</ol>

Compare the rewritten query and retrieved results with `Broken-RAG.ipynb`. This isolates query rewriting as one change; guardrails, chunking changes, and other retrieval strategies are intentionally left for later experiments.